# Токенизатор руками: UTF-8-байты, BPE и воспроизводимый словарь

Мы обучим детерминированные слияния соседних байтов, проверим encode/decode на кириллице, сериализуем правила в двоичный файл и покажем, почему тот же набор слияний в другом порядке уже является другим токенизатором.

In [ ]:
from collections import Counter
import hashlib
import platform
import struct

print('python:', platform.python_version())

def replace_pair(sequence, pair, new_id):
    result = []
    index = 0
    while index < len(sequence):
        if index + 1 < len(sequence) and (sequence[index], sequence[index + 1]) == pair:
            result.append(new_id)
            index += 2
        else:
            result.append(sequence[index])
            index += 1
    return result

def train_byte_bpe(texts, merge_count):
    sequences = [list(text.encode('utf-8')) for text in texts]
    vocabulary = {token_id: bytes([token_id]) for token_id in range(256)}
    merges = []
    for new_id in range(256, 256 + merge_count):
        counts = Counter(pair for sequence in sequences for pair in zip(sequence, sequence[1:]))
        if not counts:
            break
        # При равной частоте выбираем лексикографически меньшую пару.
        pair = min(counts, key=lambda candidate: (-counts[candidate], candidate))
        sequences = [replace_pair(sequence, pair, new_id) for sequence in sequences]
        vocabulary[new_id] = vocabulary[pair[0]] + vocabulary[pair[1]]
        merges.append((pair, new_id))
    return vocabulary, merges

def encode(text, merges):
    sequence = list(text.encode('utf-8'))
    for pair, new_id in merges:
        sequence = replace_pair(sequence, pair, new_id)
    return sequence

def decode(token_ids, vocabulary):
    return b''.join(vocabulary[token_id] for token_id in token_ids).decode('utf-8')

## 1. Обучение — это изменение словаря, а не текста

Начальные токены 0…255 обозначают отдельные байты. Каждое правило создаёт новый токен как конкатенацию байтов двух уже существующих токенов. Благодаря этому любая строка UTF-8 кодируется даже до обучения.

In [ ]:
corpus = [
    'токен токен модель',
    'модель читает токены',
    'токенизация модели',
]
vocabulary, merges = train_byte_bpe(corpus, merge_count=24)
assert len(vocabulary) == 256 + len(merges)

for index, (pair, token_id) in enumerate(merges[:8]):
    print(index, pair, '->', token_id, repr(vocabulary[token_id]))

## 2. Encode и decode образуют пару

Кодировщик обязан применять правила в том же порядке, в котором они были выучены. Декодировщик не угадывает слова: он конкатенирует байты токенов и только затем декодирует UTF-8.

In [ ]:
fixtures = corpus + ['новый токен вне корпуса', 'ASCII + кириллица: да']
for text in fixtures:
    token_ids = encode(text, merges)
    restored = decode(token_ids, vocabulary)
    assert restored == text
    print(len(text.encode('utf-8')), 'байт ->', len(token_ids), 'токенов:', token_ids[:12])

## 3. Двоичный словарь — часть модели

Для byte-BPE достаточно сохранить порядок пар: содержимое нового токена однозначно восстанавливается из предыдущих. Независимый загрузчик ниже не использует исходные Python-объекты.

In [ ]:
def dump_merges(merges):
    blob = bytearray(b'BPE1')
    blob += struct.pack('<I', len(merges))
    for (left, right), expected_id in merges:
        assert expected_id == 256 + (len(blob) - 8) // 8
        blob += struct.pack('<II', left, right)
    return bytes(blob)

def load_merges(blob):
    assert blob[:4] == b'BPE1'
    count = struct.unpack_from('<I', blob, 4)[0]
    assert len(blob) == 8 + count * 8
    vocabulary = {token_id: bytes([token_id]) for token_id in range(256)}
    merges = []
    for index in range(count):
        pair = struct.unpack_from('<II', blob, 8 + index * 8)
        token_id = 256 + index
        vocabulary[token_id] = vocabulary[pair[0]] + vocabulary[pair[1]]
        merges.append((pair, token_id))
    return vocabulary, merges

blob = dump_merges(merges)
loaded_vocabulary, loaded_merges = load_merges(blob)
assert loaded_merges == merges
assert loaded_vocabulary == vocabulary
assert decode(encode(fixtures[-1], loaded_merges), loaded_vocabulary) == fixtures[-1]
print('bytes:', len(blob), 'sha256:', hashlib.sha256(blob).hexdigest())

## 4. Намеренная поломка: правила применены в обратном порядке

Байты всё ещё восстановятся, поэтому одной проверки decode(encode(text)) недостаточно. Нужен второй инвариант — точная последовательность token_ids для контрольной строки.

In [ ]:
control_text = 'токен токен модель'
expected_ids = encode(control_text, merges)
wrong_ids = encode(control_text, list(reversed(merges)))
assert decode(wrong_ids, vocabulary) == control_text
assert wrong_ids != expected_ids
print('правильные id:', expected_ids)
print('другой порядок:', wrong_ids)

tokenizer_manifest = {
    'format': 'BPE1',
    'sha256': hashlib.sha256(blob).hexdigest(),
    'control_text_utf8_sha256': hashlib.sha256(control_text.encode('utf-8')).hexdigest(),
    'control_token_ids': expected_ids,
}
assert tokenizer_manifest['control_token_ids'] == encode(control_text, loaded_merges)

## 5. Код главы 3 из `labs/ch03`

Полная версия листингов «Один цикл слияний BPE» и «Кодирование и декодирование тремя слияниями» лежит в файле `labs/ch03/bpe_tokenizer.py`. В книге осталась сокращённая версия, а здесь файл выполняется целиком. Ячейка ниже находит каталог `labs/ch03` от текущей папки вверх и загружает файл по явному пути.

In [ ]:
import importlib.util
from pathlib import Path


def find_labs_ch03():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / 'labs' / 'ch03'
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError('labs/ch03 не найден: запустите ноутбук внутри репозитория книги')


LABS_CH03 = find_labs_ch03()


def load_lab(name):
    lab_path = LABS_CH03 / f'{name}.py'
    spec = importlib.util.spec_from_file_location(f'ch03_{name}', lab_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


print('labs:', LABS_CH03)

### 5.1. BPE по буквам на корпусе «мама», «мамы»

Механизм разделов «Строим словарь слияниями соседних фрагментов» и «Кодирование и декодирование: туда и обратно»: считаем соседние пары с весами слов, сливаем самую частую, а при ничьей берём первую по алфавиту. Потом кодируем слова выученными слияниями строго по порядку и собираем текст обратно. Функция `main()` в файле повторяет все проверки главы: три слияния вручную, словарь с `<pad>` и `<unk>`, «мамаша» с потерянной буквой «ш», проверку туда-обратно на неудобных строках и нормализацию «й».

В отличие от байтового BPE выше, здесь начальный алфавит состоит из букв. Поэтому буква вне алфавита превращается в `<unk>`, и текст уже не восстанавливается.

In [ ]:
bpe_lab = load_lab('bpe_tokenizer')
bpe_lab.main()   # все assert из файла: при ошибке ячейка упадёт

lab_merges, lab_corpus = bpe_lab.learn_merges({'мама': 2, 'мамы': 1}, steps=3)
lab_vocab = bpe_lab.build_vocab('маы', lab_merges)
print('слияния:', lab_merges)
print('словарь:', lab_vocab)
for word in ['мама', 'мамы', 'мам', 'мамаша']:
    ids = bpe_lab.encode(word, lab_merges, lab_vocab)
    print(f'{word!r:10} -> {ids} -> {bpe_lab.decode(ids, lab_vocab)!r}')

# Буквенный BPE теряет «ш», байтовый BPE из раздела 2 восстанавливает строку целиком.
assert bpe_lab.decode(bpe_lab.encode('мамаша', lab_merges, lab_vocab), lab_vocab) != 'мамаша'
assert decode(encode('мамаша', merges), vocabulary) == 'мамаша'

## Переход к BookLM

В настоящем BookLM размер embedding и выходной матрицы определяется размером словаря. Поэтому изменение порядка слияний при тех же 8192 строках матрицы не является безобидной заменой файла: номера начинают указывать на другие байтовые последовательности.

**Задание.** Добавьте специальные токены после BPE, сохраните их номера отдельно и докажите, что они никогда не возникают от обычных слияний.